In [ ]:
!pip install torch transformers sentencepiece datasets sudachipy sudachidict_core pyarrow requests

In [ ]:
!git clone https://github.com/mochiOS/ime.git
%cd ime

In [ ]:
!chmod +x ./vendor/ja/download.sh
!./vendor/ja/download.sh

In [ ]:
!curl https://sh.rustup.rs -sSf | sh -s -- -y

In [ ]:
import os

os.environ["PATH"] += ":/root/.cargo/bin"

!cargo --version
!rustc --version

In [ ]:
!cargo run --release -p engine --bin mimec -- \
    --lex vendor/ja/small_lex.csv \
    --lex vendor/ja/core_lex.csv \
    --matrix vendor/ja/matrix.def \
    -o vendor/ja/ja.mime

In [ ]:
import torch
import platform

print("Python:", platform.python_version())
print("PyTorch:", torch.__version__)
print("CUDA:", torch.cuda.is_available())

if torch.cuda.is_available():
	print("GPU:", torch.cuda.get_device_name(0))

In [ ]:
from datasets import load_dataset

DATASET_NAME = "hotchpotch/fineweb-2-edu-japanese"
DATASET_CONFIG = "sample_10BT"

corpus = load_dataset(
	DATASET_NAME,
	DATASET_CONFIG,
	split="train",
	streaming=True,
)

corpus

In [ ]:
import re

SENTENCE_SPLIT = re.compile(r"(?<=[。！？!?])")
JAPANESE = re.compile(r"[\u3040-\u30ff\u3400-\u9fff]")
URL = re.compile(r"https?://|www\.", re.IGNORECASE)


def split_sentences(text: str):
	text = text.replace("\r\n", "\n").replace("\r", "\n")
	text = re.sub(r"[ \t]+", " ", text)

	for paragraph in re.split(r"\n+", text):
		paragraph = paragraph.strip()

		if not paragraph:
			continue

		for sentence in SENTENCE_SPLIT.split(paragraph):
			sentence = sentence.strip()

			if sentence:
				yield sentence


def usable_sentence(sentence: str) -> bool:
	length = len(sentence)

	if length < 5 or length > 160:
		return False

	if URL.search(sentence):
		return False

	japanese = len(JAPANESE.findall(sentence))

	if japanese < 3:
		return False

	if japanese / length < 0.5:
		return False

	return True

In [ ]:
from sudachipy import Dictionary

sudachi = Dictionary().create()


def katakana_to_hiragana(text: str) -> str:
	return "".join(
		chr(ord(ch) - 0x60)
		if "\u30a1" <= ch <= "\u30f6"
		else ch
		for ch in text
	)


def to_reading(text: str) -> str:
	reading = "".join(
		morpheme.reading_form()
		for morpheme in sudachi.tokenize(text)
	)

	return katakana_to_hiragana(reading)

In [ ]:
MAX_SENTENCES = 100_000

sentences = []

for row in corpus:
	text = row.get("text")

	if not isinstance(text, str):
		continue

	for sentence in split_sentences(text):
		if not usable_sentence(sentence):
			continue

		sentences.append(sentence)

		if len(sentences) >= MAX_SENTENCES:
			break

	if len(sentences) >= MAX_SENTENCES:
		break

print("sentences:", len(sentences))

In [ ]:
!cargo build --release --bin candidates

In [74]:
import subprocess


class CandidateEngine:
	def __init__(
		self,
		executable: str,
		dictionary: str,
		limit: int = 16,
	):
		self.process = subprocess.Popen(
			[
				executable,
				"--dictionary",
				dictionary,
				"--limit",
				str(limit),
			],
			stdin=subprocess.PIPE,
			stdout=subprocess.PIPE,
			stderr=subprocess.PIPE,
			text=True,
			encoding="utf-8",
			bufsize=1,
		)

	def candidates(
		self,
		reading: str,
	) -> list[str]:
		if self.process.poll() is not None:
			error = self.process.stderr.read()

			raise RuntimeError(
				f"candidate engine terminated:\n{error}"
			)

		self.process.stdin.write(
			reading + "\n"
		)
		self.process.stdin.flush()

		count_line = self.process.stdout.readline()

		if not count_line:
			error = self.process.stderr.read()

			raise RuntimeError(
				f"candidate engine returned no response:\n{error}"
			)

		count = int(
			count_line.strip()
		)

		candidates = []

		for _ in range(count):
			line = self.process.stdout.readline()

			if not line:
				raise RuntimeError(
					"candidate engine terminated "
					"while returning candidates"
				)

			candidates.append(
				line.rstrip("\r\n")
			)

		return candidates

	def close(self) -> None:
		if self.process.poll() is not None:
			return

		self.process.stdin.close()
		self.process.wait()

	def __enter__(self):
		return self

	def __exit__(
		self,
		exc_type,
		exc_value,
		traceback,
	):
		self.close()

In [75]:
ENGINE = "target/release/candidates"
DICTIONARY = "vendor/ja/ja.mime"
N_BEST = 16

candidate_engine = CandidateEngine(
	ENGINE,
	DICTIONARY,
	N_BEST,
)

print(
	candidate_engine.candidates(
		"きょうはいいてんきですね。"
	)
)

['今日はいい転記ですね。', '今日はいい天気ですね。', '今日はいい転機ですね。', 'きょうはいい転記ですね。', '今日はいい転期ですね。', '今日はいい転帰ですね。', '今日はいい奠基ですね。', '今日はいい点鬼ですね。', '今日はいい天機ですね。', '今日はいい恬熈ですね。', '今日はいいてんきですね。', '経はいい転記ですね。', '卿はいい転記ですね。', '教はいい転記ですね。', '今日はいい転記ですゥね。', '今日はいい転記ですねェ。']


In [76]:
from tqdm.auto import tqdm

examples = []

for sentence in tqdm(sentences):
	reading = to_reading(sentence)

	if not reading:
		continue

	examples.append({
		"reading": reading,
		"positive": sentence,
	})

print("examples:", len(examples))

  0%|          | 0/100000 [00:00<?, ?it/s]

examples: 100000


In [ ]:
MAX_GROUPS = 10_000

training_groups = []

for example in tqdm(examples[:MAX_GROUPS]):
	positive = example["positive"]

	candidates = candidate_engine.candidates(
		example["reading"]
	)

	seen = {positive}
	negatives = []

	for candidate in candidates:
		if candidate in seen:
			continue

		seen.add(candidate)
		negatives.append(candidate)

	if not negatives:
		continue

	training_groups.append({
		"reading": example["reading"],
		"positive": positive,
		"negatives": negatives,
	})

print("groups:", len(training_groups))

  0%|          | 0/10000 [00:00<?, ?it/s]